In [2]:
# pip install scikit-clean pandas numpy sklearn

# Importo librerías

In [3]:
import numpy as np
import pandas as pd
rs=33

# Funciones testeo modelos

In [4]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler

def urlf_test_in_dfs(
    dfs, 
    dfs_names, 
    noise_levels, 
    rs=33, 
    filter = None, 
    model = RandomForestClassifier(random_state=33, n_jobs=-1),
    sc = StandardScaler(),
    k_cv=5
    ):

    # Initialize dict to store results
    res = {
        "df_name":[],
        "noise_pct":[],
        "Acc":[],
        "BalAcc":[],
        "f1_macro":[],
        "Prec_macro":[],
        "Rec_macro":[]
    }

    # Iter through dataframes
    for (df_name, df) in zip(dfs_names,dfs):

        # Extract attributes and target from df
        X = df.iloc[:,:-1].values
        y = df.iloc[:,-1].values

        # First compute baseline (no filter nor noise) results with df data
        cv = cross_validate(
            estimator=Pipeline(
                [
                    ("sc", sc),
                    ("model", model),
                ]
            ),
            X=X,
            y=y,
            scoring={
                "Acc":"accuracy",
                "BalAcc":"balanced_accuracy",
                "f1_macro":"f1_macro",
                "Prec_macro":"precision_macro",
                "Rec_macro":"recall_macro"
            },
            n_jobs=-1,
            cv=k_cv)

        # Store results
        res["df_name"].append(df_name)
        res["noise_pct"].append(-1)    # -1 means baseline with no filtering technique applied
        res["Acc"].append(cv["test_Acc"].mean())
        res["BalAcc"].append(cv["test_BalAcc"].mean())
        res["f1_macro"].append(cv["test_f1_macro"].mean())
        res["Prec_macro"].append(cv["test_Prec_macro"].mean())
        res["Rec_macro"].append(cv["test_Rec_macro"].mean())

        # Iter through noise_levels
        for nl in noise_levels:
            print(f"Processing {df_name} with noise level={nl}.")
            # Apply random uniform noise with nl as noise level
            y_noisy = urlf(y, noise_level=nl, random_state=rs)

            # Compute results without filter
            cv = cross_validate(
                estimator=Pipeline(
                    [
                        ("sc", sc),
                        ("model", model),
                    ]
                ),
                X=X,
                y=y_noisy,
                scoring={
                    "Acc":"accuracy",
                    "BalAcc":"balanced_accuracy",
                    "f1_macro":"f1_macro",
                    "Prec_macro":"precision_macro",
                    "Rec_macro":"recall_macro"
                },
                n_jobs=-1,
                cv=k_cv)

            # Store results
            res["df_name"].append(df_name+"_nf")   # nf means "not filtered"
            res["noise_pct"].append(nl)    # -1 means baseline with no filtering technique applied
            res["Acc"].append(cv["test_Acc"].mean())
            res["BalAcc"].append(cv["test_BalAcc"].mean())
            res["f1_macro"].append(cv["test_f1_macro"].mean())
            res["Prec_macro"].append(cv["test_Prec_macro"].mean())
            res["Rec_macro"].append(cv["test_Rec_macro"].mean())

            # Compute results with filter applied
            cv = cross_validate(
                estimator=Pipeline(
                    [
                        ("filter", filter),
                        ("sc", sc),
                        ("model", model),
                    ]
                ),
                X=X,
                y=y_noisy,
                scoring={
                    "Acc":"accuracy",
                    "BalAcc":"balanced_accuracy",
                    "f1_macro":"f1_macro",
                    "Prec_macro":"precision_macro",
                    "Rec_macro":"recall_macro"
                },
                n_jobs=-1,
                cv=k_cv)

            # Store results
            res["df_name"].append(df_name+"_f")   # f means "filtered"
            res["noise_pct"].append(nl)    # -1 means baseline with no filtering technique applied
            res["Acc"].append(cv["test_Acc"].mean())
            res["BalAcc"].append(cv["test_BalAcc"].mean())
            res["f1_macro"].append(cv["test_f1_macro"].mean())
            res["Prec_macro"].append(cv["test_Prec_macro"].mean())
            res["Rec_macro"].append(cv["test_Rec_macro"].mean())
        print("\n")

    return res

# Preparo datos

Para testar las técnicas a definir a continuación crearemos a partir de un dataset limpio otro con ruido:

In [5]:
from sklearn.datasets import load_breast_cancer

clean_ds = pd.concat(load_breast_cancer(return_X_y=True, as_frame=True), axis=1)
clean_ds.sample(3)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
423,13.66,19.13,89.46,575.3,0.09057,0.11470,0.09657,0.04812,0.1848,0.06181,...,25.50,101.40,708.8,0.1147,0.3167,0.3660,0.14070,0.2744,0.08839,1
272,21.75,20.99,147.30,1491.0,0.09401,0.19610,0.21950,0.10880,0.1721,0.06194,...,28.18,195.90,2384.0,0.1272,0.4725,0.5807,0.18410,0.2833,0.08858,0
79,12.86,18.00,83.19,506.3,0.09934,0.09546,0.03889,0.02315,0.1718,0.05997,...,24.82,91.88,622.1,0.1289,0.2141,0.1731,0.07926,0.2779,0.07918,1


# Testeo filtros

In [7]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

from numpy.random import randint, seed

from filters import *
from noiseData import *

# --- construir el filtro (wrapper) ---
ens_filter_sampler = EnsembleFiltering(
    estimators=[
        GaussianNB(),
        RandomForestClassifier(random_state=33),
    ],
    cv=5,
    mode="consensus",
    # threshold=0.75,
    action="remove",
    random_state=33,
)

# --- ejecutar TU función exactamente ---
noise_levels = [0.0, 0.05, 0.10, 0.20, 0.30]

res = urlf_test_in_dfs(
    dfs=[clean_ds],
    dfs_names=["ds"],
    noise_levels=noise_levels,
    rs=33,
    filter=ens_filter_sampler,  # <-- aquí el wrapper
    model=KNeighborsClassifier(),
    sc=StandardScaler(),
    k_cv=5
)

df_res = pd.DataFrame(res)
df_res

Processing ds with noise level=0.0.
Processing ds with noise level=0.05.
Processing ds with noise level=0.1.
Processing ds with noise level=0.2.
Processing ds with noise level=0.3.




,df_name,noise_pct,Acc,BalAcc,f1_macro,Prec_macro,Rec_macro
0,ds,-1.00,0.964850,0.958548,0.962058,0.967163,0.958548
1,ds_nf,0.00,0.964850,0.958548,0.962058,0.967163,0.958548
2,ds_f,0.00,0.964866,0.955741,0.961804,0.970142,0.955741
3,ds_nf,0.05,0.906816,0.895681,0.900727,0.909177,0.895681
4,ds_f,0.05,0.919127,0.905053,0.913214,0.927951,0.905053
5,ds_nf,0.10,0.854091,0.840540,0.845741,0.858456,0.840540
6,ds_f,0.10,0.864617,0.846909,0.854960,0.877714,0.846909
7,ds_nf,0.20,0.752166,0.734154,0.736803,0.755032,0.734154
8,ds_f,0.20,0.783807,0.764427,0.769265,0.796452,0.764427
9,ds_nf,0.30,0.676681,0.665554,0.663414,0.684381,0.665554
